In [1]:
import os
import sys
import time
from pathlib import Path
import pandas as pd
from IPython.display import display

JAVA_HOME = '/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home'
os.environ['JAVA_HOME'] = JAVA_HOME
os.environ['PATH'] = f"{JAVA_HOME}/bin:{os.environ['PATH']}"
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['SPARK_LOCAL_HOSTNAME'] = 'localhost'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# stop any stale Spark session before creating a new one with this notebook's runtime settings.
try:
    spark.stop()
except NameError:
    pass
except Exception:
    pass

# import Spark SQL, the streaming helper functions, and the MLlib pipeline pieces.
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, pmod, sqrt
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler


pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:.4f}')


DIR = Path('/Users/alexdevoid/Documents/Stats/ST554-HW/HW10')
FIT_DATA_PATH = DIR / 'bikeDetails_for_fit.csv'
WATCH_DIR = DIR / 'bike_stream_input'
RATE_QUERY_NAME = 'hw10_rate_table'

# start Spark session
spark = SparkSession.builder.master('local[*]').appName('hw10_structured_streaming').getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/20 16:33:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Structured Streaming with `rate`

I use Spark's `rate` source to create a stream with a timestamp and integer `value`. I add square-root and mod transformations so the memory table stores the original and derived columns.

In [ ]:

# read the rate source
rate_stream = spark.readStream.format('rate').option('rowsPerSecond', 1).load()

# add the requested square-root and mod-4 columns 
rate_transformed = (
    rate_stream
    .withColumn('sqrt_value', sqrt(col('value')))
    .withColumn('value_mod_4', pmod(col('value'), lit(4)))
)

# write the transformed rate rows to an in-memory table.
rate_query = (
    rate_transformed
    .writeStream
    .format('memory')
    .queryName(RATE_QUERY_NAME)
    .outputMode('append')
    .start()
)

# leting the query running for 30 seconds 
time.sleep(30)
rate_query.stop()

# read the full memory table after stopping the query.
rate_output = spark.sql(f'SELECT * FROM {RATE_QUERY_NAME} ORDER BY value')

# count the stored rows, then print 
rate_output_count = rate_output.count()
print(f'Rows stored in {RATE_QUERY_NAME}: {rate_output_count}')
rate_output.show(rate_output_count, truncate=False)


26/04/20 16:33:47 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/hy/304lry6j4sbbkk4_rs_vfdw80000gn/T/temporary-f92a5773-dbff-46b8-a871-61e576d60ddf. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/20 16:33:47 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/20 16:34:18 WARN DAGScheduler: Failed to cancel job group 3f26b291-a16e-42cb-a988-565e547c92bc. Cannot find active jobs for it.
26/04/20 16:34:18 WARN DAGScheduler: Failed to cancel job group 3f26b291-a16e-42cb-a988-565e547c92bc. Cannot find active jobs for it.


Rows stored in hw10_rate_table: 30
+-----------------------+-----+------------------+-----------+
|timestamp              |value|sqrt_value        |value_mod_4|
+-----------------------+-----+------------------+-----------+
|2026-04-20 16:33:48.025|0    |0.0               |0          |
|2026-04-20 16:33:49.025|1    |1.0               |1          |
|2026-04-20 16:33:50.025|2    |1.4142135623730951|2          |
|2026-04-20 16:33:51.025|3    |1.7320508075688772|3          |
|2026-04-20 16:33:52.025|4    |2.0               |0          |
|2026-04-20 16:33:53.025|5    |2.23606797749979  |1          |
|2026-04-20 16:33:54.025|6    |2.449489742783178 |2          |
|2026-04-20 16:33:55.025|7    |2.6457513110645907|3          |
|2026-04-20 16:33:56.025|8    |2.8284271247461903|0          |
|2026-04-20 16:33:57.025|9    |3.0               |1          |
|2026-04-20 16:33:58.025|10   |3.1622776601683795|2          |
|2026-04-20 16:33:59.025|11   |3.3166247903554   |3          |
|2026-04-20 16:34:00

The query runs for 30 seconds so the table has time to collect rows from the `rate` source. 
